In [0]:
# utilities.py
# Reusable helper functions for the Marathos project
# Used by: silver_notebook.py and gold notebooks

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType
from pyspark.sql.window import Window


# ─────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────

BRONZE_TABLE = "marathos.bronze.races"
SILVER_TABLE = "marathos.silver.obt"


# ─────────────────────────────────────────
# UNIT EXTRACTION
# ─────────────────────────────────────────

def get_event_unit(event_distance_length: str):
    """
    Extracts the unit from event_distance_length.
    Examples: '50km' -> 'km', '24h' -> 'h', '100mi' -> 'mi'
    Returns None if the unit is not recognised.
    """
    if event_distance_length is None:
        return None
    val = event_distance_length.lower().strip()
    if val.endswith("km"):
        return "km"
    elif val.endswith("mi"):
        return "mi"
    elif val.endswith("h"):
        return "h"
    elif val.endswith("d"):
        return "d"
    return None

get_event_unit_udf = F.udf(get_event_unit, StringType())


def get_performance_unit(athlete_performance: str):
    """
    Determines whether athlete_performance is a time (HH:MM:SS) or distance (km).
    Returns 'h' for time format, 'km' for decimal/integer format.
    Returns None if it cannot be determined.
    """
    if athlete_performance is None:
        return None
    perf = athlete_performance.strip()
    if ":" in perf:
        return "h"
    try:
        float(perf)
        return "km"
    except ValueError:
        return None

get_performance_unit_udf = F.udf(get_performance_unit, StringType())


def is_valid_unit_combination(event_unit: str, performance_unit: str) -> bool:
    """
    Validates that the event unit / performance unit combination is logical:
      - event = km or mi  →  performance must be a time (h)
      - event = h         →  performance must be a distance (km)
    """
    if event_unit is None or performance_unit is None:
        return False
    if event_unit in ("km", "mi") and performance_unit == "h":
        return True
    if event_unit == "h" and performance_unit == "km":
        return True
    return False

is_valid_unit_combination_udf = F.udf(is_valid_unit_combination)


# ─────────────────────────────────────────
# TIME CONVERSION
# ─────────────────────────────────────────

def time_to_seconds(time_str: str):
    """
    Converts a time string to total seconds (int).
    Handles formats: 'H:MM:SS', 'HH:MM:SS', 'D day, H:MM:SS'
    Examples:
      '2:34:10'         -> 9250
      '1 day, 3:00:00'  -> 97200
    Returns None if conversion fails.
    """
    if time_str is None:
        return None
    try:
        s = time_str.strip()
        extra_days = 0
        if "day" in s:
            parts = s.split(",")
            extra_days = int(parts[0].strip().split(" ")[0])
            s = parts[1].strip()
        hms = s.split(":")
        if len(hms) == 3:
            h, m, sec = int(hms[0]), int(hms[1]), int(float(hms[2]))
            return extra_days * 86400 + h * 3600 + m * 60 + sec
        return None
    except Exception:
        return None

time_to_seconds_udf = F.udf(time_to_seconds, IntegerType())


def seconds_to_hms(total_seconds: int) -> str:
    """
    Converts total seconds back to a readable string 'HH:MM:SS'.
    Useful for display purposes in Gold/Dashboard.
    """
    if total_seconds is None:
        return None
    h = total_seconds // 3600
    m = (total_seconds % 3600) // 60
    s = total_seconds % 60
    return f"{h:02}:{m:02}:{s:02}"

seconds_to_hms_udf = F.udf(seconds_to_hms, StringType())


# ─────────────────────────────────────────
# SURROGATE KEY GENERATION (sha2)
# ─────────────────────────────────────────

def add_hash_id(df: DataFrame, source_cols: list, id_col: str) -> DataFrame:
    """
    Generates a stable surrogate key using sha2 (256-bit hash).
    Better than dense_rank() for streaming pipelines — the ID is
    deterministic and does not require a full table scan.

    Parameters:
      df          - input DataFrame
      source_cols - list of columns to hash together (e.g. ['event_name'])
      id_col      - name of the new id column (e.g. 'event_id')

    Example:
      df = add_hash_id(df, ["event_name"], "event_id")
      df = add_hash_id(df, ["athlete_id"], "athlete_key")
      df = add_hash_id(df, ["year_of_event"], "year_id")
    """
    concat_col = F.concat_ws("||", *[F.col(c).cast("string") for c in source_cols])
    return df.withColumn(id_col, F.sha2(concat_col, 256))


# ─────────────────────────────────────────
# DATA QUALITY HELPERS
# ─────────────────────────────────────────

def null_summary(df: DataFrame, spark) -> DataFrame:
    """
    Returns a DataFrame with null counts and percentages per column.
    Useful during EDA steps.
    """
    total = df.count()
    null_counts = [(col, df.filter(F.col(col).isNull()).count()) for col in df.columns]
    rows = [(col, cnt, round(cnt / total * 100, 2)) for col, cnt in null_counts]
    return spark.createDataFrame(rows, ["column", "null_count", "null_pct"])


def value_counts(df: DataFrame, col: str, top_n: int = 20) -> DataFrame:
    """
    Returns the most frequent values in a column.
    Replaces pandas value_counts() for use with PySpark DataFrames.
    """
    return (
        df.groupBy(col)
          .count()
          .orderBy(F.desc("count"))
          .limit(top_n)
    )